In [0]:
%sql
Select * from workspace.ecommerce.events limit(10)

In [0]:
revenue_trends_df = spark.sql("""
SELECT
  date_trunc('month', event_time) AS month,
  ROUND(SUM(price), 2) AS total_revenue
FROM workspace.ecommerce.events
WHERE event_type = 'purchase'
GROUP BY month
ORDER BY month
""")

display(revenue_trends_df)


In [0]:
funnel_df = spark.sql("""
SELECT
  event_type,
  COUNT(DISTINCT user_id) AS users
FROM workspace.ecommerce.events
WHERE event_type IN ('view', 'cart', 'purchase')
GROUP BY event_type
ORDER BY users DESC
""")

display(funnel_df)


Databricks visualization. Run in Databricks to view.

In [0]:
top_products_df = spark.sql("""
SELECT
  product_id,
  ROUND(SUM(price), 2) AS total_revenue,
  COUNT(*) AS purchase_count
FROM workspace.ecommerce.events
WHERE event_type = 'purchase'
GROUP BY product_id
ORDER BY total_revenue DESC
LIMIT 10
""")

display(top_products_df)

Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col

filtered_df = revenue_trends_df.filter(
    (col("month") >= "2019-01-01") & (col("month") <= "2020-12-31")
)

display(filtered_df)


In [0]:
%sql
-- Customer tiers
SELECT
  CASE WHEN cnt >= 10 THEN 'VIP'
       WHEN cnt >= 5 THEN 'Loyal'
       ELSE 'Regular' END as tier,
  COUNT(*) as customers,
  AVG(total_spent) as avg_ltv
FROM (
  SELECT user_id, COUNT(*) cnt, SUM(price) total_spent
  FROM workspace.ecommerce.events
  WHERE event_type = 'purchase'
  GROUP BY user_id
)
GROUP BY tier

In [0]:
%sql
-- Revenue with 7-day moving average
WITH daily AS (
  SELECT
    date(event_time) AS event_date,
    SUM(price) AS rev
  FROM workspace.ecommerce.events
  WHERE event_type = 'purchase'
  GROUP BY date(event_time)
)
SELECT
  event_date,
  rev,
  AVG(rev) OVER (ORDER BY event_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW) AS ma7
FROM daily
ORDER BY event_date;